# 4a. Adaptation Analysis

How do animals adapt when the stimulus distribution shifts?

This is the central behavioural result for Aim 1. The distribution shift
trajectory (Uniform → Hard-A → Hard-B → Hard-A) provides three
qualitatively different transitions:

1. **First novel** — first shift the animal experiences
2. **Second experienced** — second shift, with prior experience of change
3. **Familiar return** — returning to a previously experienced distribution

For each transition we characterise:
- Stat trajectories aligned to the shift
- Update matrix evolution across the shift
- Psychometric curve changes
- Recovery dynamics (exponential fits, sessions-to-criterion)
- Whether the winning model's predictions match actual adaptation

**Depends on:** 0a (data loaded), 3a/3c (model assignments)

In [ ]:
%matplotlib inline
from shared_setup import *

from analysis.adaptation import (
    detect_all_manipulations,
    classify_shift_type,
    group_shifts_by_type,
    adaptation_trajectory,
    fit_recovery_curve,
    compare_phases,
    compute_shift_magnitude,
    compute_convergence_metrics,
    aggregate_trajectories,
)

## 1. Configuration

In [ ]:
TRACK_STATS = ['accuracy', 'pse', 'slope', 'recency', 'win_stay_rate', 'side_bias']
RECOVERY_STATS = ['accuracy', 'pse', 'recency']
N_BASELINE = 5       # sessions to use as expert baseline reference
N_EARLY_POST = 5     # sessions to define "early post-shift" phase

## 2. Detect and Classify Shifts

In [ ]:
experiment, load_info = load_data()

shift_animals = {}  # animal_id -> {animal, manips, classified, ...}

for aid in experiment.animal_ids:
    animal = experiment.get_animal(aid)
    manips = detect_all_manipulations(animal, stage=STAGE)
    if not manips:
        print(f"{aid}: no shifts detected")
        continue

    classified = classify_shift_type(manips)
    all_sessions = animal.get_sessions(stage=STAGE)

    shift_animals[aid] = {
        'animal': animal,
        'manips': classified,
        'all_sessions': all_sessions,
    }
    for m in classified:
        print(
            f"{aid}: {m['shift_type']} at session {m['global_session_idx']} "
            f"({m['details'].get('before', '?')} → {m['details'].get('after', '?')})"
        )

print(f"\n{len(shift_animals)} animals with distribution shifts")

## 3. Group Shifts by Type

Group all shifts across all animals by type, so we can later
compare adaptation dynamics across types.

In [ ]:
# Build the structure expected by group_shifts_by_type
animals_for_grouping = {}
for aid, info in shift_animals.items():
    animals_for_grouping[aid] = {
        'animal': info['animal'],
        'manips': info['manips'],
        'stage': STAGE,
    }

shifts_by_type = group_shifts_by_type(animals_for_grouping)

for stype, entries in shifts_by_type.items():
    animals_in_type = [e['animal_id'] for e in entries]
    print(f"{stype}: {len(entries)} shifts from {animals_in_type}")

## 4. Per-Animal Adaptation Trajectories

For each animal's first shift: stat trajectories aligned to the
shift point, with baseline mean ± SD bands.

In [ ]:
per_animal_results = {}

for aid, info in shift_animals.items():
    first_shift = info['manips'][0]
    all_sessions = info['all_sessions']
    shift_idx = first_shift['session_idx']

    baseline = all_sessions[max(0, shift_idx - N_BASELINE):shift_idx]
    post = all_sessions[shift_idx:]

    if len(baseline) < 2 or len(post) < 2:
        print(f"{aid}: skipping (baseline={len(baseline)}, post={len(post)})")
        continue

    traj = adaptation_trajectory(baseline, post, stats=TRACK_STATS)
    traj['animal_id'] = aid

    per_animal_results[aid] = {
        'trajectory': traj,
        'baseline': baseline,
        'post': post,
        'shift': first_shift,
    }

print(f"\n{len(per_animal_results)} animals with sufficient data for analysis")

In [ ]:
# ── Plot per-animal trajectories ────────────────────────────────────────────
for aid, res in per_animal_results.items():
    traj = res['trajectory']
    shift = res['shift']
    shift_label = (
        f"{shift['details'].get('before', '?')} → "
        f"{shift['details'].get('after', '?')}"
    )

    n_stats = len(TRACK_STATS)
    n_cols = min(3, n_stats)
    n_rows = int(np.ceil(n_stats / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(5 * n_cols, 3.5 * n_rows), sharex=True)
    axes = np.atleast_2d(axes)

    for idx, stat in enumerate(TRACK_STATS):
        if stat not in traj.columns:
            continue
        row, col = divmod(idx, n_cols)
        ax = axes[row, col]

        bl_mask = traj['phase'] == 'baseline'
        post_mask = traj['phase'] == 'post'

        ax.plot(traj.loc[bl_mask, 'relative_session'],
                traj.loc[bl_mask, stat], 'o-', ms=4, color='steelblue')
        ax.plot(traj.loc[post_mask, 'relative_session'],
                traj.loc[post_mask, stat], 'o-', ms=4, color='darkorange')

        # Baseline band
        bl_mean = traj[f'baseline_{stat}_mean'].iloc[0]
        bl_std = traj[f'baseline_{stat}_std'].iloc[0]
        if not np.isnan(bl_mean):
            ax.axhline(bl_mean, color='steelblue', ls='--', lw=0.8, alpha=0.5)
            ax.axhspan(bl_mean - bl_std, bl_mean + bl_std,
                        color='steelblue', alpha=0.08)

        ax.axvline(0, color='black', ls=':', lw=0.8)
        ax.set_title(stat, fontsize=10)

    for idx in range(n_stats, n_rows * n_cols):
        row, col = divmod(idx, n_cols)
        axes[row, col].set_visible(False)

    axes[-1, 0].set_xlabel('Sessions relative to shift')
    fig.suptitle(f"{aid}: {shift_label} ({shift['shift_type']})",
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 5. Update Matrix Evolution Across the Shift

Side-by-side UMs at key phases. The UM structure (win-stay for BE,
stripe for SC) should change character across the shift.

In [ ]:
for aid, res in per_animal_results.items():
    baseline = res['baseline']
    post = res['post']
    shift = res['shift']

    phases = {}

    # Expert baseline
    bl = baseline[-N_BASELINE:] if len(baseline) >= N_BASELINE else baseline
    um, _, um_info = compute_update_matrix_from_sessions(bl, method='pool')
    phases[f'Expert\n({um_info["n_sessions"]}s)'] = um

    # Early post
    n_early = min(N_EARLY_POST, len(post))
    if n_early >= 2:
        um, _, um_info = compute_update_matrix_from_sessions(
            post[:n_early], method='pool',
        )
        phases[f'Early post\n({um_info["n_sessions"]}s)'] = um

    # Late post
    if len(post) > N_EARLY_POST:
        late = post[N_EARLY_POST:]
        um, _, um_info = compute_update_matrix_from_sessions(
            late, method='pool',
        )
        phases[f'Late post\n({um_info["n_sessions"]}s)'] = um

    if len(phases) >= 2:
        fig, axes = plot_phase_update_matrices(
            phases,
            suptitle=(
                f"{aid}: {shift['details'].get('before', '?')} → "
                f"{shift['details'].get('after', '?')}"
            ),
        )
        plt.tight_layout()
        plt.show()

## 6. Psychometric Comparison Across the Shift

In [ ]:
for aid, res in per_animal_results.items():
    baseline = res['baseline']
    post = res['post']
    shift = res['shift']

    groups = {'Expert baseline': baseline[-N_BASELINE:]}
    if len(post) >= 3:
        groups['Early post (1–3)'] = post[:3]
    if len(post) >= 6:
        groups['Mid post'] = post[3:min(6, len(post))]
    if len(post) > 6:
        groups['Late post'] = post[-3:]

    fig, ax, infos = plot_psychometric_overlay(
        groups,
        mode='pooled',
        n_bootstrap=200,
        show_ci=True,
        title=(
            f"{aid}: {shift['details'].get('before', '?')} → "
            f"{shift['details'].get('after', '?')}"
        ),
    )
    plt.show()

## 7. Recovery Dynamics

Fit exponential recovery curves and compute convergence metrics.

In [ ]:
convergence_rows = []

for aid, res in per_animal_results.items():
    baseline = res['baseline']
    post = res['post']
    shift = res['shift']

    conv = compute_convergence_metrics(baseline, post, stats=RECOVERY_STATS)
    conv['animal_id'] = aid
    conv['shift_type'] = shift['shift_type']
    convergence_rows.append(conv)

    print(f"\n{aid} ({shift['shift_type']}):")
    for _, row in conv.iterrows():
        if row['converged']:
            print(
                f"  {row['stat']:>12s}: τ={row['tau']:.1f} sessions, "
                f"R²={row['r_squared']:.2f}, "
                f"sessions to criterion={row['sessions_to_criterion']:.0f}"
            )
        else:
            print(f"  {row['stat']:>12s}: fit did not converge")

convergence_df = pd.concat(convergence_rows, ignore_index=True)

## 8. Phase Comparisons: Statistical Tests

In [ ]:
for aid, res in per_animal_results.items():
    baseline = res['baseline']
    post = res['post']
    shift = res['shift']

    print(f"\n=== {aid} ({shift['shift_type']}) ===")

    if len(post) >= N_EARLY_POST:
        comp = compare_phases(
            baseline[-N_BASELINE:], post[:N_EARLY_POST],
            stats=TRACK_STATS, test='mannwhitneyu',
        )
        print("  Baseline vs Early post:")
        for _, row in comp.iterrows():
            sig = '*' if row['p_value'] < 0.05 else ' '
            print(
                f"    {row['stat']:>15s}: Δ={row['cliffs_delta']:+.2f} "
                f"p={row['p_value']:.3f} {sig}"
            )

    # Shift magnitude
    for metric in ['accuracy_drop', 'pse_shift']:
        mag = compute_shift_magnitude(baseline, post, metric=metric)
        print(f"  {metric}: {mag['value']:+.3f}")

## 9. Cross-Shift-Type Comparison

Do the three transition types differ in adaptation speed or pattern?
Group all shifts by type and compare recovery dynamics.

In [ ]:
if len(shifts_by_type) > 1:
    for stype, entries in shifts_by_type.items():
        type_conv = convergence_df[convergence_df['shift_type'] == stype]
        if type_conv.empty:
            continue
        print(f"\n=== {stype} (n={len(entries)} shifts) ===")
        for stat in RECOVERY_STATS:
            stat_data = type_conv[type_conv['stat'] == stat]
            converged = stat_data[stat_data['converged']]
            if len(converged) > 0:
                tau_vals = converged['tau'].values
                stc_vals = converged['sessions_to_criterion'].dropna().values
                print(
                    f"  {stat}: τ={np.mean(tau_vals):.1f}±{np.std(tau_vals):.1f}, "
                    f"sessions-to-criterion="
                    f"{np.mean(stc_vals):.1f}±{np.std(stc_vals):.1f} "
                    f"(n={len(converged)} converged)"
                )
            else:
                print(f"  {stat}: no converged fits")
else:
    print("Only one shift type — cross-type comparison not applicable yet")

## 10. Group-Level Trajectories

Mean ± SEM across animals, aligned to the shift.

In [ ]:
all_trajectories = [
    res['trajectory'] for res in per_animal_results.values()
]

if len(all_trajectories) > 1:
    agg = aggregate_trajectories(
        all_trajectories, stats=TRACK_STATS, session_range=(-10, 15),
    )

    n_cols = min(3, len(TRACK_STATS))
    n_rows = int(np.ceil(len(TRACK_STATS) / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(5 * n_cols, 3.5 * n_rows), sharex=True)
    axes = np.atleast_2d(axes)

    for idx, stat in enumerate(TRACK_STATS):
        row, col = divmod(idx, n_cols)
        ax = axes[row, col]

        stat_data = agg[agg['stat'] == stat].sort_values('relative_session')
        if stat_data.empty:
            ax.set_title(f'{stat} (no data)', fontsize=10)
            continue

        x = stat_data['relative_session'].values
        y = stat_data['mean'].values
        yerr = stat_data['sem'].values

        bl_mask = x < 0
        post_mask = x >= 0

        ax.errorbar(x[bl_mask], y[bl_mask], yerr=yerr[bl_mask],
                    fmt='o-', ms=4, color='steelblue', capsize=2)
        ax.errorbar(x[post_mask], y[post_mask], yerr=yerr[post_mask],
                    fmt='o-', ms=4, color='darkorange', capsize=2)
        ax.axvline(0, color='black', ls=':', lw=0.8)
        ax.set_title(stat, fontsize=10)

    for idx in range(len(TRACK_STATS), n_rows * n_cols):
        row, col = divmod(idx, n_cols)
        axes[row, col].set_visible(False)

    axes[-1, 0].set_xlabel('Sessions relative to shift')
    fig.suptitle(
        f'Group mean (n={len(all_trajectories)} animals)',
        fontsize=12, fontweight='bold',
    )
    plt.tight_layout()
    plt.show()
else:
    print("Need >1 animal with shifts for group-level plot")

## 11. Model Prediction Overlay

Does the winning model's predicted adaptation match real behaviour?
Load model assignments from 3a/3c and simulate the shift using
best-fit expert parameters.

In [ ]:
# Load model assignments if available
from pathlib import Path
assignments_path = Path('../results/model_assignments.csv')

if assignments_path.exists():
    assignments = pd.read_csv(assignments_path)
    print(f"Loaded model assignments for {len(assignments)} animals")
    print(assignments[['animal_id', 'winner']].to_string(index=False))
else:
    print(
        "No model assignments found at results/model_assignments.csv. "
        "Run notebooks 3a/3c first. Skipping model prediction overlay."
    )
    assignments = None

In [ ]:
# Overlay model predictions on real trajectories
# This section requires:
# 1. Model assignments (from 3a/3c)
# 2. Best-fit parameters (from 3a)
# 3. Stimulus distribution sampler (from analysis.stimulus_distribution)
#
# The comparison: simulate the winning model with expert-phase parameters
# through the same shift, plot predicted stat trajectory alongside real.

if assignments is not None:
    from analysis.stimulus_distribution import sample_distribution
    from analysis.grid_search import _simulate_um

    # TODO: implement per-animal model prediction overlay
    # For each animal with a shift and a model assignment:
    #   1. Load best-fit params from 3a results
    #   2. Simulate shift sequence with those params
    #   3. Compute per-session stats from simulated data
    #   4. Overlay on real trajectory from section 4
    print("Model prediction overlay: implementation pending")
    print("Requires best-fit params from 3a cluster results")

## 12. Save Results

In [ ]:
# import pickle
# results = {
#     'per_animal': per_animal_results,
#     'convergence': convergence_df,
#     'shifts_by_type': {k: len(v) for k, v in shifts_by_type.items()},
# }
# with open('../results/adaptation/adaptation_results.pkl', 'wb') as f:
#     pickle.dump(results, f)
# print("Saved to results/adaptation/adaptation_results.pkl")